# SQL Injection — Hands-On

**DBPRA · TU Berlin**

You play the attacker. You are given a small SQLite database and a Python function that builds SQL by string concatenation.
Your job: cause as much damage as you can.

There are two parts:

1. **Extract data the query was not supposed to return.**
2. **Do damage, be creative.**

Both rely on the same vulnerability — string concatenation — but the second part requires a small change to the Python wrapper, which we explain when we get there.

> Run the cells from top to bottom. If you trash the database in Part 2 and want a fresh start, run `reset_db()`.

In [1]:
import sqlite3
import warnings
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning, module="pandas")

SCHEMA_AND_DATA = """
DROP TABLE IF EXISTS customer;
CREATE TABLE customer (
  c_w_id         INTEGER NOT NULL,
  c_d_id         INTEGER NOT NULL,
  c_id           INTEGER NOT NULL,
  c_discount     REAL,
  c_credit       TEXT,
  c_last         TEXT,
  c_first        TEXT,
  c_credit_lim   REAL,
  c_balance      REAL,
  c_ytd_payment  REAL,
  c_payment_cnt  INTEGER,
  c_delivery_cnt INTEGER,
  c_street_1     TEXT,
  c_street_2     TEXT,
  c_city         TEXT,
  c_state        TEXT,
  c_zip          TEXT,
  c_phone        TEXT,
  c_since        TEXT,
  c_middle       TEXT,
  c_data         TEXT
);
INSERT INTO customer VALUES
  (1,1,1,0.10,'GC','Smith','John',1000.00,500.00,100.00,1,1,'123 Main St','Anytown','Anytown','CA','12345','123-456-7890','2022-01-01','X','Customer 1'),
  (1,1,2,0.20,'GC','Johnson','Jane',2000.00,1000.00,200.00,2,2,'456 Elm St','Othertown','Othertown','NY','67890','987-654-3210','2022-01-02','Y','Customer 2'),
  (1,2,1,0.30,'GC','Williams','Bob',3000.00,1500.00,300.00,3,3,'789 Oak St','Thistown','Thistown','TX','34567','555-123-4567','2022-01-03','Z','Customer 3'),
  (2,1,1,0.40,'GC','Jones','Alice',4000.00,2000.00,400.00,4,4,'901 Maple St','Thatown','Thatown','FL','90123','111-222-3333','2022-01-04','A','Customer 4'),
  (2,1,2,0.50,'GC','Brown','Mike',5000.00,2500.00,500.00,5,5,'234 Pine St','Thiscity','Thiscity','IL','45678','444-555-6666','2022-01-05','B','Customer 5'),
  (2,2,1,0.60,'GC','Davis','Emily',6000.00,3000.00,600.00,6,6,'567 Cedar St','Thatcity','Thatcity','OH','78901','777-888-9999','2022-01-06','C','Customer 6');
"""

con = sqlite3.connect(":memory:")

def reset_db():
    """Wipe and re-create the customer table. Run this if you trash the DB."""
    con.executescript(SCHEMA_AND_DATA)
    print("Database reset.")

reset_db()
pd.read_sql("SELECT c_id, c_first, c_last, c_city, c_discount FROM customer", con)


Database reset.


,c_id,c_first,c_last,c_city,c_discount
0,1,John,Smith,Anytown,0.1
1,2,Jane,Johnson,Othertown,0.2
2,1,Bob,Williams,Thistown,0.3
3,1,Alice,Jones,Thatown,0.4
4,2,Mike,Brown,Thiscity,0.5
5,1,Emily,Davis,Thatcity,0.6


The `executescript` method runs **any number of statements** separated by semicolons — stacking a destructive statement after the `SELECT` fires.

In [12]:
def search_customers(city: str):
    """Vulnerable AND multi-statement."""
    query = (
        "SELECT c_last, c_first FROM customer "
        "WHERE c_city LIKE '" + city + "' AND c_discount >= 0.3;"
    )
    print("--- script sent to DB ---")
    print(query)
    print("-------------------------")
    con.executescript(query)
    # executescript() throws away result rows; show DB state instead so you
    # can see what your payload did.
    tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con)
    print("tables now in DB:", list(tables["name"]))
    if "customer" in list(tables["name"]):
        return pd.read_sql("SELECT c_id, c_first, c_last, c_city, c_discount FROM customer", con)
    return None

# Legitimate use still works (the SELECT runs, just no rows come back):
search_customers("Thatown")


--- script sent to DB ---
SELECT c_last, c_first FROM customer WHERE c_city LIKE 'Thatown' AND c_discount >= 0.3;
-------------------------
tables now in DB: ['customer']


,c_id,c_first,c_last,c_city,c_discount
0,1,John,Smith,Anytown,0.1
1,2,Jane,Johnson,Othertown,0.2
2,1,Bob,Williams,Thistown,0.3
3,1,Alice,Jones,Thatown,0.4
4,2,Mike,Brown,Thiscity,0.5
5,1,Emily,Davis,Thatcity,0.6


### Your task

Find a payload for `city` that does as much damage as you can.
Between attempts you can inspect state with:

```python
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con)
pd.read_sql("SELECT * FROM customer", con)
```


In [13]:
# Your payload here.
reset_db()
payload = "Thistown"
search_customers(payload)

Database reset.
--- script sent to DB ---
SELECT c_last, c_first FROM customer WHERE c_city LIKE 'Thistown' AND c_discount >= 0.3;
-------------------------
tables now in DB: ['customer']


,c_id,c_first,c_last,c_city,c_discount
0,1,John,Smith,Anytown,0.1
1,2,Jane,Johnson,Othertown,0.2
2,1,Bob,Williams,Thistown,0.3
3,1,Alice,Jones,Thatown,0.4
4,2,Mike,Brown,Thiscity,0.5
5,1,Emily,Davis,Thatcity,0.6
